In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Run this cell first every time you open a new Colab session.
# Clones the repo (data and artifacts included) and installs required packages.
import os, sys, importlib

REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
REPO_PATH = "/content/packt-modern-time-series"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

# Stay in student_notebooks so Path().resolve().parent resolves to repo root
os.chdir(f"{REPO_PATH}/student_notebooks")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# torch: Colab already has it (GPU build). Install CPU-only if missing (local use).
if importlib.util.find_spec("torch") is None:
    os.system("pip install -q torch --index-url https://download.pytorch.org/whl/cpu")

# neuralforecast: NHITS and NeuralForecast wrapper
os.system("pip install -q neuralforecast")

print(f"✓ Setup complete — {os.getcwd()}")

# Module 6 — Deep Learning with NHITS
**Type:** [Code With Me]  
**Time:** 40 minutes  
**Job:** Implement a global neural forecasting model (NHITS). Compare it against the LightGBM and baseline floors. Be honest about what deep learning adds — and what it costs.

---
## 6.1 — Setup
**[Watch Only]**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts, build_leaderboard
from src.schemas import validate_forecast

print("Setup complete.")

---
## 6.2 — Load Panel and Micro Subset
**[Watch Only]**

In [ ]:
panel = load_checkpoint("03_validated_panel")

top_series = (
    panel.groupby("unique_id")["y"]
    .sum()
    .sort_values(ascending=False)
    .head(MICRO_SUBSET_N)
    .index
)
micro = panel[panel["unique_id"].isin(top_series)].copy()

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 6.3 — Why NHITS?
**[Watch Only]**

NHITS (Neural Hierarchical Interpolation for Time Series) is a global neural model designed specifically for multi-step ahead forecasting. Published at AAAI 2023.

**What it does differently from LightGBM:**
- Learns across multiple temporal scales simultaneously using hierarchical pooling
- Produces the full 28-step horizon in a single forward pass — no recursive error accumulation
- Captures non-linear interactions that tree-based models with fixed lag windows miss

**What it costs:**
- Training time is not parallelizable in the same way as LightGBM — it is a neural network with gradient updates
- Hyperparameter sensitivity is higher — `max_steps`, `input_size`, and learning rate all affect stability
- Interpretability is lower — you cannot extract feature importances the way you can from a tree model

**The production question:**  
NHITS is worth the cost when your demand patterns have complex multi-scale dependencies that lag features cannot capture — long promotional build-ups, multi-week seasonality stacking, or cross-series structural patterns. For stable weekly seasonal demand, LightGBM often wins on the ROI calculation.

---
## 6.4 — Load NHITS Parameters
**[Watch Only]**

In [ ]:
params_path = PARAMS_DIR / "nhits_tuned.json"
with open(params_path) as f:
    nhits_params = json.load(f)

nhits_params = {k: v for k, v in nhits_params.items() if not k.startswith("_")}
nhits_params["random_seed"] = RANDOM_SEED
nhits_params["h"] = HORIZON

print("NHITS parameters:")
for k, v in nhits_params.items():
    print(f"  {k:<30}: {v}")
print(f"\n  Runtime lever: max_steps = {nhits_params['max_steps']}")
print(f"  Reduce to 200 if training exceeds 90 seconds.")

---
## 6.5 — Configure NHITS
**[Code With Me — 2 lines]**

Fill in `input_size` and `max_steps` from the `nhits_params` dict loaded above.

In [ ]:
nhits_model = NHITS(
    h=nhits_params["h"],
    input_size=__FILL_IN__,                       # nhits_params["input_size"]
    max_steps=__FILL_IN__,                        # nhits_params["max_steps"] — primary runtime lever
    val_check_steps=nhits_params["val_check_steps"],
    early_stop_patience_steps=nhits_params["early_stop_patience_steps"],
    n_freq_downsample=nhits_params["n_freq_downsample"],
    learning_rate=nhits_params["learning_rate"],
    batch_size=nhits_params["batch_size"],
    random_seed=nhits_params["random_seed"],
    enable_progress_bar=nhits_params["enable_progress_bar"],
    enable_model_summary=nhits_params["enable_model_summary"],
)

nf = NeuralForecast(models=[nhits_model], freq="D")

print(f"NeuralForecast configured.")
print(f"  Model       : NHITS")
print(f"  h           : {nhits_params['h']}")
print(f"  input_size  : {nhits_params['input_size']}")
print(f"  max_steps   : {nhits_params['max_steps']}")

**Expected output:**
```
NeuralForecast configured.
  Model       : NHITS
  h           : 28
  input_size  : 56
  max_steps   : 300
```

---
## 6.6 — Run Cross-Validation on the Micro Subset
**[Watch Only]**

> **⚠ Runtime warning:** Target < 60 seconds. Hard ceiling 90 seconds. If this cell approaches 90 seconds, interrupt it (`I, I` in Jupyter / square stop button in Colab) and run the Red Path in 6.7.

In [ ]:
%%time
# NHITS cross-validation on micro subset (50 series)
# Target runtime: < 60 seconds | Hard ceiling: 90 seconds
# If this cell exceeds 90 seconds: interrupt (I, I) and run the Red Path in 6.7

cv_dl_micro = nf.cross_validation(
    df=micro,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=REFIT,
)

print(f"NHITS CV complete. Shape: {cv_dl_micro.shape}")
print(f"Columns: {list(cv_dl_micro.columns)}")
print(cv_dl_micro.head(3).to_string())
GREEN_PATH_SUCCEEDED = True

**Expected output:**
```
NHITS CV complete. Shape: (4200, 5)
Columns: ['ds', 'cutoff', 'y', 'NHITS']
```

---
## 6.7 — Red Path Recovery
**[Watch Only]**

In [ ]:
# 🔴 RED PATH — run this cell only if 6.6 timed out or failed

# from src.checkpointing import load_checkpoint
# cv_dl_micro = None
# GREEN_PATH_SUCCEEDED = False
# print("Red Path active — micro NHITS results unavailable. Full-subset artifact loaded in 6.11.")

---
## 6.8 — Reshape and Add Conformal Intervals
**[Code With Me — 2 lines]**

Same conformal pattern as Module 5. Fill in the residual calculation and the 10th percentile. The 90th percentile is provided — match the pattern.

In [ ]:
def reshape_neuralforecast_cv(cv_df: pd.DataFrame, model_col: str, stage: str) -> pd.DataFrame:
    """
    Reshape NeuralForecast wide CV output to forecast schema.
    Adds conformal prediction intervals from in-sample residuals.
    """
    df = cv_df.reset_index().copy()
    df = df.rename(columns={model_col: "y_hat"})
    df["model"] = "NHITS"
    df["stage"] = stage
    df["y_hat"] = df["y_hat"].clip(lower=0)

    residuals = __FILL_IN__                        # df["y"] - df["y_hat"]
    lo_offset = __FILL_IN__                        # np.percentile(residuals, 10)
    hi_offset = np.percentile(residuals, 90)

    df["lo_80"] = (df["y_hat"] + lo_offset).clip(lower=0)
    df["hi_80"] = (df["y_hat"] + hi_offset).clip(lower=0)

    return df[["unique_id", "ds", "y", "model", "y_hat", "lo_80", "hi_80", "cutoff", "stage"]]


if cv_dl_micro is not None:
    dl_micro = reshape_neuralforecast_cv(cv_dl_micro, model_col="NHITS", stage="dl")
    dl_micro_validated = validate_forecast(dl_micro, artifact_name="06_dl_micro")
    print(f"Reshaped DL forecasts: {dl_micro.shape}")
    print(f"Columns: {list(dl_micro.columns)}")
else:
    print("Green Path not available. Skipping micro reshape — using full artifact in 6.11.")
    dl_micro_validated = None

**Expected output (if Green Path succeeded):**
```
Reshaped DL forecasts: (4200, 9)
Columns: ['unique_id', 'ds', 'y', 'model', 'y_hat', 'lo_80', 'hi_80', 'cutoff', 'stage']
```

---
## 6.9 — Plot: NHITS vs LightGBM vs SeasonalNaive
**[Watch Only]**

In [ ]:
if dl_micro_validated is not None:
    sample_uid = top_series[0]
    sample_cut = dl_micro_validated["cutoff"].unique()[-1]
    actuals    = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]
    plot_start = sample_cut - pd.Timedelta(days=42)
    nhits_fcast = dl_micro_validated[
        (dl_micro_validated["unique_id"] == sample_uid) &
        (dl_micro_validated["cutoff"]    == sample_cut)
    ].set_index("ds")
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(actuals[actuals.index >= plot_start].index,
            actuals[actuals.index >= plot_start].values,
            color="#333", linewidth=1.0, label="Actual", zorder=3)
    ax.plot(nhits_fcast.index, nhits_fcast["y_hat"],
            color="#F4511E", linewidth=1.5, linestyle="--", label="NHITS", zorder=4)
    ax.fill_between(nhits_fcast.index, nhits_fcast["lo_80"], nhits_fcast["hi_80"],
                    alpha=0.15, color="#F4511E", label="NHITS 80% interval")
    try:
        ml_micro_path = ARTIFACT_DIR / "05_ml_micro_forecasts.parquet"
        ml_micro = pd.read_parquet(ml_micro_path)
        lgbm_fcast = ml_micro[
            (ml_micro["unique_id"] == sample_uid) &
            (ml_micro["cutoff"]    == sample_cut)
        ].set_index("ds")
        if len(lgbm_fcast) > 0:
            ax.plot(lgbm_fcast.index, lgbm_fcast["y_hat"],
                    color="#7B1FA2", linewidth=1.2, linestyle=":", label="LightGBM", zorder=3)
    except Exception:
        pass
    ax.axvline(sample_cut, color="#999", linestyle=":", linewidth=1)
    ax.set_title(f"NHITS vs LightGBM — {sample_uid} (Window 3)", fontsize=11)
    ax.set_ylabel("Units sold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("Green Path not available — plot skipped. Visual comparison available in Module 8.")

**Expected output (if Green Path succeeded):** NHITS forecast with shaded 80% interval and LightGBM dotted overlay.

---
## 6.10 — Score the Micro DL Forecasts
**[Code With Me — 1 line]**

Fill in the `score_forecasts` call. Use `f"micro_{MICRO_SUBSET_N}"` as the `subset_name`.

In [ ]:
if dl_micro_validated is not None:
    dl_micro_scores = __FILL_IN__  # score_forecasts(dl_micro_validated, subset_name=f"micro_{MICRO_SUBSET_N}")

    wmape_row  = dl_micro_scores[dl_micro_scores["metric"] == "wMAPE"].iloc[0]
    iscore_row = dl_micro_scores[dl_micro_scores["metric"] == "IntervalScore_80"].iloc[0]

    print(f"NHITS on micro subset ({MICRO_SUBSET_N} series):")
    print(f"  wMAPE          : {wmape_row['score']:.4f}")
    print(f"  Interval Score : {iscore_row['score']:.4f}")
else:
    print("Micro scoring skipped — Red Path was taken. Full-subset scores in 6.11.")

**Expected output (if Green Path succeeded):**
```
NHITS on micro subset (50 series):
  wMAPE          : 0.XXXX
  Interval Score : X.XXXX
```

---
## 6.11 — Load Full-Subset DL Forecasts and Update Leaderboard
**[Watch Only]**

In [ ]:
# 🔴 RED PATH (standard) — always load precomputed full-subset DL forecasts
dl_full = load_checkpoint("06_dl_forecasts")

dl_full_scores       = score_forecasts(dl_full,       subset_name=f"workshop_{WORKSHOP_SUBSET_N}")
baseline_full        = load_checkpoint("04_baseline_forecasts")
baseline_full_scores = score_forecasts(baseline_full, subset_name=f"workshop_{WORKSHOP_SUBSET_N}")
ml_full              = load_checkpoint("05_ml_forecasts")
ml_full_scores       = score_forecasts(ml_full,       subset_name=f"workshop_{WORKSHOP_SUBSET_N}")

leaderboard_6 = build_leaderboard([baseline_full_scores, ml_full_scores, dl_full_scores])

print("Six-model leaderboard (wMAPE):")
if "wMAPE" in leaderboard_6.columns:
    print(leaderboard_6[["model", "wMAPE"]].dropna().sort_values("wMAPE").to_string(index=False))
else:
    print(leaderboard_6.to_string(index=False))

**Expected output:**
```
Six-model leaderboard (wMAPE):
          model    wMAPE
       LightGBM   0.XXXX
          NHITS   0.XXXX
        AutoETS   0.XXXX
Chronos-t5-mini   0.XXXX
  SeasonalNaive   0.XXXX
          Naive   0.XXXX
```

---
## 6.12 — Save the Micro DL Artifact
**[Watch Only]**

In [ ]:
if dl_micro_validated is not None:
    micro_dl_path = ARTIFACT_DIR / "06_dl_micro_forecasts.parquet"
    dl_micro_validated.to_parquet(micro_dl_path, index=False)
    print(f"  ✓ Micro DL artifact saved : {micro_dl_path.name} ({len(dl_micro_validated):,} rows)")
print(f"  ✓ Full DL artifact loaded : 06_dl_forecasts.parquet ({len(dl_full):,} rows)")

---
## 6.13 — The Enterprise Cliff
**[Watch Only]**

We trained NHITS in under 90 seconds on 50 series. In production, the cost profile is completely different.

**Training time scales with data, not series count:**  
NHITS processes every time step as a training sample. At 100,000 SKUs × 3 years of daily history, a single training run takes hours on CPU. This is a GPU job in production — and the infrastructure cost needs to be justified by the accuracy margin over LightGBM.

**Retraining cadence is a system design decision:**  
LightGBM can be retrained daily in minutes. NHITS cannot. In practice, neural forecasting models are retrained weekly or monthly and serve predictions from a frozen checkpoint in between. This introduces model staleness risk — especially around holidays or demand shocks.

**The wMAPE gap tells you whether to proceed:**  
If NHITS beats LightGBM by 2+ wMAPE points on your portfolio, the GPU compute cost is likely justified. If the gap is less than 1 point, LightGBM is the production choice — it is faster, cheaper, and more interpretable.

Look at your leaderboard. Make the call.